# GridSense
## A real-data-first analysis of electricity reliability in Addis Ababa

**Student:** Lisan A  
**Country:** Ethiopia  
**Course:** Kujenga Final Project

Electricity outages are not an abstract infrastructure issue for me. In Addis Ababa, they affect studying, internet access, small businesses, work meetings, food storage, and daily planning. Many people only know an outage is coming after it has already disrupted their day.

This project asks a focused local question:

> **What can public data tell us about electricity reliability in Addis Ababa, and what data would be needed to build a responsible outage-risk prediction system?**

I avoid using synthetic data as the main dataset. Instead, I use public, non-personal evidence and build a model-ready path for future verified reports.


## 1. Why this fits Kujenga

Kujenga asks us to choose a real-world question, find data, clean it, apply a course technique or a more advanced method, visualize the results, and tell a compelling story. My problem is local: electricity reliability in Addis Ababa.

This notebook uses:

- **Regression** from the happiness lesson: a trend line through real Enterprise Survey outage data.
- **Model thinking** from the SIR/SEIR lessons: risk changes over time and depends on conditions.
- **Hypothesis-testing discipline** from the runners lesson: I do not run invalid tests when the sample is too small.
- **Network/PageRank thinking** from the PageRank lesson: future outage reports could identify central co-outage areas.
- **Responsible AI engineering:** clear limits, no personal data, no synthetic data as main evidence.


In [1]:
from pathlib import Path
import csv, json, math

ROOT = Path('..') if Path('../data').exists() else Path('.')

def read_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

context = read_csv(ROOT / 'data/raw/addis_ababa_reliability_context.csv')
wb = read_csv(ROOT / 'data/raw/ethiopia_enterprise_outage_indicators.csv')
events = read_csv(ROOT / 'data/raw/verified_public_outage_evidence.csv')
schema = read_csv(ROOT / 'data/raw/model_ready_event_schema.csv')

print(f'Local public evidence rows: {len(context)}')
print(f'Enterprise Survey indicator rows: {len(wb)}')
print(f'Verified public evidence log rows: {len(events)}')
print(f'Model-ready schema fields: {len(schema)}')


Local public evidence rows: 13
Enterprise Survey indicator rows: 7
Verified public evidence log rows: 4
Model-ready schema fields: 12


## 2. Data honesty ladder

The meeting reminder said: **if possible, avoid synthetic datasets**. I take that seriously. The table below separates what is real from what is only a future data plan.

| Layer | Status | Used for |
|---|---|---|
| Public Addis Ababa reliability evidence | Real/public | Main story and analysis |
| Ethiopia Enterprise Survey indicators | Real/public | National/business context |
| Verified public evidence log | Real/public but small | Timeline and motivation |
| Community report template | Future data collection | Safe scaling plan |
| Synthetic data | Not used as main dataset | Excluded from this final version |

This is important because an impressive model trained on invented rows would not prove real Addis Ababa outage patterns.


![Problem context](../reports/figures/problem_context.svg)

The first story is simple: electricity access is high, but reliability is still a challenge. That makes this a local reliability problem, not just an access problem.


In [1]:
metrics = {row['metric']: float(row['value']) for row in context}
summary = {
    'electricity_access_percent': metrics['electricity_access'],
    'mv_interruptions_per_year': metrics['medium_voltage_line_interruptions'],
    'mv_interruption_hours_per_year': metrics['medium_voltage_interruption_duration'],
    'average_hours_per_interruption': round(metrics['medium_voltage_interruption_duration'] / metrics['medium_voltage_line_interruptions'], 2),
    'estimated_unresolved_issues_count': int(metrics['outage_related_problems_identified'] * (1 - metrics['outage_related_problems_resolved']/100))
}
for key, value in summary.items():
    print(f'{key}: {value}')


electricity_access_percent: 99.0
mv_interruptions_per_year: 882.0
mv_interruption_hours_per_year: 2103.0
average_hours_per_interruption: 2.38
estimated_unresolved_issues_count: 11500


## 3. What the public data says

The public evidence suggests four important points:

1. Addis Ababa has very high electricity access.
2. Reliability remains a measurable problem: the baseline includes hundreds of medium-voltage interruptions and thousands of interruption-hours.
3. A later public issue audit reported many outage-related problems in the capital.
4. Infrastructure upgrades are already underway, meaning the problem is active and policy-relevant.

![Reliability baseline](../reports/figures/reliability_baseline.svg)


## 4. Enterprise Survey context

Because local event-level outage data is limited, I also use Ethiopia-level Enterprise Survey indicators as context. These are not a substitute for Addis Ababa event logs, but they show that electricity reliability has been a serious business constraint.

![World Bank outage trend](../reports/figures/worldbank_outage_trend.svg)


In [1]:
# Kujenga-style regression by hand on the real Enterprise Survey outage/month series.
# This follows the regression lesson: fit y = m*x + k by minimizing squared error.
rows = [(int(r['year']), float(r['value'])) for r in wb if r['indicator_code'] == 'IC.ELC.OUTG']
xs = [x for x, y in rows]
ys = [y for x, y in rows]
xbar = sum(xs) / len(xs)
ybar = sum(ys) / len(ys)
m = sum((x-xbar)*(y-ybar) for x,y in rows) / sum((x-xbar)**2 for x in xs)
k = ybar - m*xbar
sse = sum((y - (m*x + k))**2 for x,y in rows)
print(f'Regression line: outages_per_month = {m:.3f} * year + {k:.3f}')
print(f'Sum of squared errors: {sse:.3f}')
print('Interpretation: descriptive trend only, because n=3.')


Regression line: outages_per_month = 0.724 * year + -1448.600
Sum of squared errors: 0.309
Interpretation: descriptive trend only, because n=3.


### Regression interpretation

The slope is positive, meaning the Enterprise Survey series increased between 2006 and 2015. I do **not** use this as a forecast model because there are only three data points. The value of including it is to show the Kujenga regression method and to support the story that outage reliability has mattered economically.


## 5. Outage Severity Index

A useful local tool should not only say whether an outage happened. It should help prioritise severity.

I define a transparent decision-support index:

\[
	ext{Severity} = 0.30D + 0.25F + 0.25I + 0.20U
\]

Where:

- \(D\) = normalized interruption duration evidence,
- \(F\) = normalized interruption frequency evidence,
- \(I\) = normalized issue-count evidence,
- \(U\) = unresolved issue share.

This is **not** a ground-truth label. It is a transparent engineering score for prioritisation.

![Severity components](../reports/figures/severity_components.svg)


In [1]:
components = read_csv(ROOT / 'reports/tables/outage_severity_index_components.csv')
score = sum(float(r['weighted_score']) for r in components)
print(f'Evidence-based severity score (0 to 1 scale, approximate): {score:.3f}')
for r in components:
    print(f"- {r['component']}: weight={r['weight']}, weighted_score={r['weighted_score']}")


Evidence-based severity score (0 to 1 scale, approximate): 0.279
- MV interruptions per year: weight=0.25, weighted_score=0.0088
- MV interruption hours per year: weight=0.3, weighted_score=0.0252
- Identified outage-related issues: weight=0.25, weighted_score=0.25
- Unresolved issue share: weight=0.2, weighted_score=0.0004


## 6. Future ML model: formulas and responsible design

The project can become a real machine-learning system after enough verified local reports are collected. I do not train the final model on synthetic data, but I define the exact ML formulation.

### Classification model: outage risk

For a future report row \(i\), define:

\[
x_i = [	ext{hour}, 	ext{day}, 	ext{sub-city}, 	ext{sefer}, 	ext{rain}, 	ext{wind}, 	ext{planned notice}, 	ext{season}, 	ext{source confidence}]
\]

The logistic regression model would be:

\[
P(y_i=1|x_i)=\sigma(eta_0+eta^Tx_i)
\]

where \(y_i=1\) means an outage occurs in the prediction window and

\[
\sigma(z)=rac{1}{1+e^{-z}}
\]

The loss function is binary cross entropy:

\[
L(eta)=-\sum_i[y_i\log(p_i)+(1-y_i)\log(1-p_i)]
\]

### Regression model: outage duration

For outage duration:

\[
\hat{d_i}=eta_0+eta^Tx_i
\]

and the model minimizes:

\[
SSE=\sum_i(d_i-\hat{d_i})^2
\]

This connects directly to the Kujenga regression lesson.

![ML pipeline](../reports/figures/ml_pipeline.svg)


## 7. Why I do not run a t-test yet

The runners lesson taught us that a t-test is powerful when we have two real groups with enough observations. For example, with enough verified outage reports I would test:

\[
H_0: \mu_{rainy} = \mu_{clear}
\]

\[
H_1: \mu_{rainy} > \mu_{clear}
\]

where \(\mu\) is mean outage duration.

The t-statistic would be:

\[
t=rac{ar{x}_1-ar{x}_2}{\sqrt{s_1^2/n_1+s_2^2/n_2}}
\]

But with the current public evidence log, there are not enough comparable event-level duration observations. So I do not report a fake p-value. That is part of responsible data science.


## 8. Model-ready dataset design

The table below is the exact schema I would use to collect future verified reports safely. It avoids personal data.


In [1]:
for row in schema:
    print(f"{row['feature']:24s} | {row['type']:16s} | {row['why_it_matters']}")


timestamp                | datetime         | Outage risk can depend on hour, day, season, and peak demand.
sub_city                 | category         | Location is needed to learn spatial reliability patterns.
sefer_or_landmark        | category         | Local place names make reports useful for Addis Ababa residents.
outage_start_time        | datetime         | Needed to compute duration and time features.
outage_end_time          | datetime         | Needed to compute duration; can be blank if ongoing.
duration_hours           | number           | Target variable for duration regression.
planned_notice           | binary           | Separates announced maintenance from unexpected outages.
weather_condition        | category         | Weather may affect faults and restoration time.
rain_mm                  | number           | Numerical weather feature for ML.
wind_speed_kmh           | number           | Wind can affect overhead lines and service restoration.
source_type         

## 9. Ethics and privacy

A local reporting system must be useful without exposing people. My future report template avoids names, phone numbers, exact household addresses, and private identifiers. It only needs approximate location, time, duration, whether the outage was planned, and optional non-personal notes.

The model should also avoid blaming communities. If one area has many reports, it may mean more people reported there, not necessarily that the grid is worse there. This is why confidence scores and official notices matter.


## 10. Conclusion

This project began with a local problem I care about: electricity outages in Addis Ababa. The analysis shows that the issue is not simply electricity access; it is reliability and predictability.

The project's main contribution is not a flashy model trained on synthetic data. It is a responsible data science foundation:

- public evidence about Addis Ababa reliability,
- economic context from Enterprise Survey outage indicators,
- a transparent severity index,
- a model-ready event schema,
- a privacy-safe community report template,
- and clear ML formulas for future prediction once enough real reports exist.

### What I would do next

1. Collect verified community reports for 4-6 weeks.
2. Add real hourly weather from Open-Meteo or a meteorological source.
3. Add official planned interruption notices.
4. Train logistic regression for outage risk and regression for duration.
5. Deploy the Streamlit dashboard as a local planning tool.

That is the path from a Kujenga final project to a useful Addis Ababa data product.
